In [6]:
# --- IMPORT STATEMENTS ---

import numpy as np
import pandas as pd
import re
from astropy.coordinates import SkyCoord
import astropy.units as u

In [7]:
# --- GLOBAL VARIABLES ---

LINUX_DIR = "/home/aimee/mphys"
DATA_DIR = f'{LINUX_DIR}/data'
FILE_DIR = f'{DATA_DIR}/Paladini2003_synthetic.tsv'

In [8]:
# --- FUNCTION DEFINITIONS ---

In [9]:
def get_Paladini_data():

    # get raw data  
    with open(FILE_DIR, 'r') as file: # this file is so ugly istg
        data = []
        for i, line in enumerate(file):
            if i > 37: # skip headers
                arr_raw = re.split(r'[ |]+', line)
                arr = []
                for j in arr_raw:
                    if (j != '') & (j != '|') & (j != '**') & (j != '<'): # this is worse than DA Green catalogue omg
                        arr.append(j)
                # cut array past index 17 (don't need comments, but do need consistent dimensions)
                arr = arr[0:17]
                if '\n' not in arr:
                    data.append(arr)

    # Extract useful info
    ra_arr, dec_arr, ang_diameter_arr, ang_diam_err_arr = [], [], [], [] # add more if needed
    for row in data:
        # RA
        ra_deg = 360/24 * (float(row[5]) + float(row[6])/60 + float(row[7])/3600) # HOURS, minutes, seconds --> degrees
        ra_arr.append(ra_deg) # deg
    
        # Dec
        if float(row[8]) < 0:
            multiplier = -1
        else:
            multiplier = +1
        dec_deg = multiplier * (np.abs(float(row[8])) + float(row[9])/60 + float(row[10])/3600) # degrees, minutes, seconds --> degrees
        dec_arr.append(float(dec_deg)) # deg
    
        # Angular diameter and error
        ang_diameter_arr.append(float(row[13])) # arcmin
        ang_diam_err_arr.append(float(row[14])) # arcmin


    # Convert ra and dec to galactic coords
    l_arr, b_arr = [], []
    for ra, dec, in zip(ra_arr, dec_arr):
        eq_coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame='icrs')
        galactic_coord = eq_coord.galactic
        l_arr.append(float(galactic_coord.l.value))
        b_arr.append(float(galactic_coord.b.value))
    
    l_cut, b_cut, ang_diameter_cut, ang_diam_err_cut = [], [], [], []
    for l, b, ang_diameter, ang_diam_err in zip(l_arr, b_arr, ang_diameter_arr, ang_diam_err_arr):
        if ang_diameter > 5: # only keep sources > 5 arcmin
            l_cut.append(l)
            b_cut.append(b)
            ang_diameter_cut.append(ang_diameter)
            ang_diam_err_cut.append(ang_diam_err)

    return l_cut, b_cut, ang_diameter_cut, ang_diam_err_cut


In [10]:
l_arr, b_arr, ang_diameter_arr, ang_diam_err_arr = get_Paladini_data()

for l, b, ang_diam, ang_diam_err in zip(l_arr, b_arr, ang_diameter_arr, ang_diam_err_arr):
    print(l, b, ang_diam, ang_diam_err)

# print(l_arr) # deg
# print(b_arr) # deg
# print(ang_diameter_arr) # arcmin
# print(ang_diam_err_arr) # arcmin

0.09846448273803338 -0.00019280745842426288 5.9 0.5
0.19873426293437904 -0.10024877963716863 10.7 0.5
0.19869162583233854 -0.0002526298344488912 9.2 0.5
0.39865912430946376 -0.8004723557516322 7.0 2.3
2.3987435734525935 1.399567236686018 7.9 3.1
2.8987130349518857 -0.0002480207183448683 6.0 2.5
3.2986979846634283 -0.10033296415301114 6.9 2.3
4.99867197301628 0.29979659580908674 17.0 1.8
5.298556399020071 -1.1002367811806617 9.2 2.3
5.898599464055603 -0.40043612161762165 6.0 2.0
5.998715293594503 -1.3002747097931469 8.9 2.2
5.998705037910113 -1.2000994974962835 8.5 2.8
6.098671875190018 -0.10023075068880495 5.9 1.5
6.398570647742769 -0.5001231376899037 6.5 1.6
6.498686371165514 0.0995363399962144 7.5 1.9
6.598681690413033 -0.300195602549694 6.7 1.9
6.598677544439852 0.09989549892688354 11.8 6.2
6.6986629342832265 -0.2001373401432835 6.3 0.6
6.8984742695713095 -2.100168127756981 16.5 5.2
6.9986675357661605 -0.3001919594501749 5.2 1.7
6.998700965249696 -0.20008358682476785 5.6 1.7
7.29861